# Transfer learning from a pretrained network

Training an image classifier from scratch takes many thousands of labelled photos. **Transfer learning** avoids that: we take a network that has already been trained on a large dataset and reuse what it learned for our own task. In this tutorial we tell cats from dogs with only 100 training photos of each. We start from ResNet-18, a classical network trained on ImageNet, a collection of over a million photos in 1000 classes, and put a small quantum model on top of it.

This tutorial builds on the [QML tutorial](./QMLtutorial.ipynb), which introduces the encoding, the ansatz and the expectation values we use here. As there, we write vectors underlined, such as the angles $\underline{x}$ and the ansatz parameters $\underline{w}$.

<p align="center"><img src="../../_static/qml_tl_pipeline.svg" alt="The transfer learning pipeline. A photo enters ResNet-18, which is frozen and drawn in grey, and comes out as 512 features. A trained linear layer with weights M and offsets b reduces them to five angles x. The quantum circuit, trained, encodes the angles and applies the ansatz W(w), and each of its five qubits is measured, giving the expectation values Z0 to Z4. The trained head with weights v and offset c combines them into the prediction y-hat. A dashed arrow from the circuit back to the linear layer marks where the gradient from the parameter-shift rule is passed on." width="880"></p>

Read it from left to right.

- ResNet-18 turns each photo into 512 numbers, its **features**. We keep the network **frozen**: its own parameters never change.
- A linear layer, which we train, reduces the 512 features to five angles $\underline{x}$, one per qubit.
- The quantum circuit encodes the five angles and applies an ansatz with trainable parameters $\underline{w}$. We measure every qubit, which gives five expectation values $\langle Z_0 \rangle, \dots, \langle Z_4 \rangle$.
- A small classical layer, the **head**, which we also train, combines the five expectation values into one prediction $\hat{y}$ between $-1$ and $+1$.

The three trained parts have 2581 parameters between them, against four in the QML tutorial. That is too many for the optimizer we used there: COBYLA would keep a group of 2582 points and compute the loss at all of them in every iteration. So we train with gradients instead. The linear layer and the head are ordinary classical code, so JAX computes their derivatives automatically. The circuit cannot be differentiated automatically, so we compute its derivatives with the **parameter-shift rule**, from extra runs of the circuit; `qrisp.qml` provides it as `parameter_shift`. All three parts are then updated together, in the same training step. The dashed arrow in the figure marks where those derivatives are passed back into the linear layer.

## Requirements

Besides Qrisp, this tutorial needs PyTorch and torchvision for the pretrained network and the photos, optax for the optimizer, and certifi for the downloads. We ran it with Python 3.11 and these versions:

```
pip install torch==2.4.1 torchvision==0.19.1 optax==0.2.8 certifi
```

On the first run, torchvision downloads the ResNet-18 weights, about 45 MB, and the Oxford-IIIT Pet dataset, about 800 MB, which takes up about 1.6 GB on disk.

In [ ]:
import os
import tempfile
import time
from pathlib import Path

import certifi
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import OxfordIIITPet
import matplotlib.pyplot as plt
from IPython.display import clear_output

import jax
import optax
import jax.numpy as jnp

from qrisp import QuantumFloat
from qrisp.operators import Z
from qrisp.qml import Ansatz, Encoding, expectation, parameter_shift, save_model, load_model
from qrisp.jasp import jaspify

# The python.org installer on macOS ships no certificates, so downloads would fail; use certifi's.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

## The data

We use the [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/), about 7400 photos of 37 breeds of cats and dogs. Besides its breed, every photo carries a second label that says only cat or dog, and that is the one we use. torchvision keeps it in `_bin_labels`; `_labels` holds the breeds.

We label cats $+1$ and dogs $-1$, the range of an expectation value, as in the QML tutorial. We take 100 random photos of each class from the dataset's training part, and another 100 of each from its test part, which the model never sees during training.

ResNet-18 expects its input prepared the way its own training photos were: resized, cropped to 224 by 224 pixels, and with the colours normalised. `weights.transforms()` does this for us.

Below are two photos of each class, as they come from the dataset.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

train_data = OxfordIIITPet(root=".", split="trainval",
                           target_types="binary-category", download=True)
test_data = OxfordIIITPet(root=".", split="test",
                          target_types="binary-category", download=True)


def take(dataset, per_class, seed):
    rng = np.random.default_rng(seed)
    labels = np.array(dataset._bin_labels)
    chosen = np.concatenate([rng.permutation(np.where(labels == c)[0])[:per_class]
                             for c in (0, 1)])
    images = torch.stack([preprocess(dataset[i][0].convert("RGB")) for i in chosen])
    return images, np.where(labels[chosen] == 0, 1.0, -1.0)


per_class = 100
train_images, y_train = take(train_data, per_class, seed=0)
test_images, y_test = take(test_data, per_class, seed=1)

print("train images:", tuple(train_images.shape))

In [ ]:
labels = np.array(train_data._bin_labels)
shown = np.concatenate([np.where(labels == c)[0][:2] for c in (0, 1)])
names = {0: "cat  (+1)", 1: "dog  (−1)"}

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, i in zip(axes, shown):
    image, label = train_data[i]
    ax.imshow(image.convert("RGB"))
    ax.set_title(names[label])
    ax.axis("off")
fig.tight_layout()
plt.show()

## Features from the frozen network

ResNet-18 was trained to sort photos into the 1000 ImageNet classes. Its layers turn the pixels, step by step, into more and more general descriptions of the photo, and its last layer turns the final 512 numbers into one score per class. We have no use for those classes, so we replace the last layer with `torch.nn.Identity`, which passes its input on unchanged. The network then outputs the 512 numbers themselves: the features.

What remains of the network is often called the **backbone**. We freeze it. `eval()` switches it from training to prediction, and `requires_grad_(False)` marks every one of its parameters as fixed. A frozen network always gives the same features for the same photo, so we compute the features of all our photos once, here. We named the weights explicitly at the start, `IMAGENET1K_V1`, so that the saved model at the end can ask for exactly these weights again. From then on, the model sees only an array of numbers.

In [ ]:
backbone = resnet18(weights=weights)
backbone.fc = torch.nn.Identity()
backbone.eval()
for parameter in backbone.parameters():
    parameter.requires_grad_(False)

with torch.no_grad():
    train_features = backbone(train_images).numpy()
    test_features = backbone(test_images).numpy()

print("features        :", train_features.shape)
print("trainable in backbone:", sum(p.requires_grad for p in backbone.parameters()))

## Standardising the features

The 512 features come in different ranges, so before they enter the linear layer we put them on a common scale. From every feature we subtract its mean and divide by its standard deviation, so that over the training photos each feature has mean $0$ and standard deviation $1$. This is called standardising. As with `minmax_scale` in the QML tutorial, the means and standard deviations come from the training photos alone and are applied unchanged to the test photos, so nothing about the test photos reaches the model during training.

We do not turn the features into angles here. That is the job of the linear layer, which is trained together with the rest of the model.

In [ ]:
num_qubits = 5

mean = train_features.mean(axis=0)
# The small constant avoids dividing by zero for a feature that is the same on every training photo.
std = train_features.std(axis=0) + 1e-8

F_train = jnp.array((train_features - mean) / std)
F_test = jnp.array((test_features - mean) / std)

print("features into the linear layer:", tuple(F_train.shape))

## The model

The model has three trained parts, and the cell below sets up each of them.

The linear layer computes five weighted sums of the 512 features $f_j$, with a matrix $M$ of $512 \times 5$ weights and five offsets $\underline{b}$. A weighted sum can be any number, but angle encoding needs angles in $[0, \pi]$, for the reason we saw in the scaling section of the QML tutorial: within that range every value gets its own state, and nothing wraps around. So we pass each sum through the sigmoid function $\sigma(s) = 1/(1 + e^{-s})$, which squeezes any number into the range from $0$ to $1$, and multiply by $\pi$:

$$x_i = \pi\,\sigma\Big(\sum_{j} f_j M_{ji} + b_i\Big), \qquad i = 0, \dots, 4.$$

The circuit is the one from the QML tutorial, widened to five qubits: angle encoding, then one layer of `Ansatz.real_amplitudes`. On five qubits, the CX gates form a ring: each qubit controls the next, and the last qubit controls the first. The ansatz has ten parameters $\underline{w}$, five before the CX gates and five after.

<p align="center"><img src="../../_static/qml_tl_circuit.svg" alt="The five-qubit circuit. Each qubit i starts in |0>, is rotated by R_Y(x_i) in the encoding, marked in green, and by R_Y(w_i) at the start of the ansatz, marked in blue. Five CX gates follow in a ring: qubit 0 controls qubit 1, 1 controls 2, 2 controls 3, 3 controls 4, and 4 controls 0. Each qubit then gets R_Y(w_5) to R_Y(w_9) and is measured in Z, giving the expectation values Z0 to Z4." width="720"></p>

We measure every qubit, so for one photo the circuit gives five expectation values $q_i = \langle Z_i \rangle$. As in the QML tutorial's section on a different readout, passing a list of observables to `expectation` gives all five. We use `precision=0.15` here instead of `0.05`. The number of measurements grows with the inverse square of the precision, so this takes nine times fewer. Training runs the circuit about five thousand times, so we accept noisier estimates in exchange for speed.

The head combines the five expectation values into one prediction,

$$\hat{y} = \tanh\big(\underline{v} \cdot \underline{q} + c\big),$$

with five weights $\underline{v}$ and one offset $c$. The function $\tanh$ keeps the prediction between $-1$ and $+1$, the range of the labels. We start with the linear layer at small random weights, every ansatz parameter at $0.5$, and the head as a plain average of the five expectation values: every weight $1/5$ and $c = 0$. With every weight at $1$, the sum of five values between $-1$ and $+1$ would often push $\tanh$ close to $\pm 1$, where it is almost flat. Every derivative of the loss passes through $\tanh$, so training would start slowly.

In [ ]:
encoding = Encoding.angle(n_features=num_qubits)
ansatz = Ansatz.real_amplitudes(num_qubits=num_qubits, layers=1)


def circuit(x, w):
    qf = QuantumFloat(num_qubits)
    encoding.apply(qf, x)
    ansatz.apply(qf, w)
    return qf


observables = [Z(i) for i in range(num_qubits)]
precision = 0.15
quantum_block = expectation(circuit, observables, precision=precision)


def angles(f, p):
    return jnp.pi * jax.nn.sigmoid(f @ p["M"] + p["b"])


def head(q, p):
    return jnp.tanh(jnp.dot(q, p["v"]) + p["c"])


parameters = {
    "M": 0.05 * jax.random.normal(jax.random.PRNGKey(0), (F_train.shape[1], num_qubits)),
    "b": jnp.zeros(num_qubits),
    "w": jnp.full(ansatz.num_parameters, 0.5),
    "v": jnp.ones(num_qubits) / num_qubits,
    "c": jnp.array(0.0),
}

print("trainable parameters:", sum(int(v.size) for v in parameters.values()))
print("  linear layer :", int(parameters["M"].size + parameters["b"].size))
print("  ansatz       :", int(parameters["w"].size))
print("  head         :", int(parameters["v"].size + parameters["c"].size))

2581 parameters are trained in total: 2565 in the linear layer ($512 \times 5$ weights and 5 offsets), 10 in the ansatz and 6 in the head. None of them belong to ResNet-18, which stays frozen.

## Training with gradients

Training moves every parameter a small step in the direction that lowers the loss. That direction is given by the gradient: the derivative of the loss with respect to each parameter. JAX computes derivatives of classical code automatically, but it cannot differentiate through the circuit, because the expectation values come from sampling. For the circuit we use the parameter-shift rule instead.

### The parameter-shift rule

Take one ansatz parameter $w_k$ and hold everything else fixed. It enters the circuit through a single rotation $R_Y(w_k)$, whose matrix contains only $\cos(w_k/2)$ and $\sin(w_k/2)$. An expectation value multiplies two such entries, and the products of two half angles combine into $\cos w_k$ and $\sin w_k$. So as a function of $w_k$ alone, every expectation value has the form

$$\langle Z \rangle(w_k) = \alpha + \beta\cos w_k + \gamma\sin w_k,$$

where the numbers $\alpha$, $\beta$ and $\gamma$ depend on the angles and the other parameters, but not on $w_k$. Evaluating this function a quarter turn to either side of $w_k$ and taking half the difference gives

$$\frac{\langle Z \rangle(w_k + \pi/2) - \langle Z \rangle(w_k - \pi/2)}{2} = -\beta\sin w_k + \gamma\cos w_k,$$

which is exactly the derivative of $\langle Z \rangle$ at $w_k$. Two extra runs of the circuit therefore give the derivative with respect to one parameter, exact apart from the sampling noise in the two values. The same holds for each angle $x_i$, since it too enters through a single $R_Y$ rotation. `parameter_shift` from `qrisp.qml` applies this rule to every parameter and every angle.

<p align="center"><img src="../../_static/qml_parameter_shift.svg" alt="The parameter-shift rule. An expectation value plotted against one parameter w_k is a shifted cosine curve. At the current value w_k, a tangent line shows the derivative. Two dots mark the curve at w_k - pi/2 and w_k + pi/2. Half the difference of their heights equals the slope of the tangent." width="560"></p>

### Putting the gradient together

The loss for one photo with label $y$ is $(\hat{y} - y)^2$. The chain rule splits its gradient into steps that follow the model backwards, from right to left in the figure below.

- JAX differentiates the head directly. This gives the gradient for $\underline{v}$ and $c$, and also how the loss changes with each of the five expectation values $q_i$.
- The parameter-shift rule gives how each $q_i$ changes with each ansatz parameter $w_k$ and each angle $x_i$. Combined with the previous step, this gives the gradient for $\underline{w}$ and for the five angles.
- JAX carries the gradient for the five angles back through the sigmoid and the weighted sums to all 2565 parameters of $M$ and $\underline{b}$. This step needs no runs of the circuit.

<p align="center"><img src="../../_static/qml_tl_gradient.svg" alt="Where the gradient comes from. The forward pass runs left to right: features f, linear layer M and b, angles x, circuit with parameters w, expectation values q, head v and c, loss. The backward pass runs right to left beneath it: automatic differentiation through the head gives the gradient for v, c and q; the parameter-shift rule gives the gradient for w and x, at 1 + 2 times 5 + 2 times 10 = 31 runs of the circuit per photo; automatic differentiation through the linear layer gives the gradient for M and b with no runs of the circuit." width="820"></p>

So for each photo we run the circuit once at the current values and twice more for each of the 5 angles and the 10 ansatz parameters: $1 + 2 \times 5 + 2 \times 10 = 31$ runs. In each run, `expectation` estimates the five expectation values with a separate round of measurements for each. The count does not depend on the size of the linear layer: with ten times as many features, it would still be 31.

In [ ]:
value_and_derivatives = parameter_shift(circuit, observables, precision=precision, argnums=(0, 1))


@jaspify(terminal_sampling=True)
def circuit_with_derivatives(x, w):
    """The five expectation values, and their derivatives with respect to x and w."""
    return value_and_derivatives(x, w)


@jaspify(terminal_sampling=True)
def q_batch(X, w):
    """The expectation values for a whole batch of angle vectors."""
    return jnp.stack([quantum_block(X[i], w) for i in range(X.shape[0])])


def gradient(f, target, p):
    """The loss and its gradient for one photo."""
    x, pullback = jax.vjp(lambda params: angles(f, params), p)
    q, (dq_dx, dq_dw) = circuit_with_derivatives(x, p["w"])

    def loss_from_q(q, params):
        return (head(q, params) - target) ** 2

    value, (g_q, g_head) = jax.value_and_grad(loss_from_q, argnums=(0, 1))(q, p)
    g_angles = pullback(dq_dx @ g_q)[0]

    return float(value), {"M": g_angles["M"], "b": g_angles["b"],
                          "w": dq_dw @ g_q,
                          "v": g_head["v"], "c": g_head["c"]}

`parameter_shift` takes the same circuit and observables as `expectation` and returns a function that gives the five expectation values together with their derivatives. With `argnums=(0, 1)` it differentiates with respect to both arguments of `circuit`, the angles $\underline{x}$ and the parameters $\underline{w}$. We call it inside one function under `@jaspify`, `circuit_with_derivatives`, so that all 31 runs for a photo happen in one call. Every separate call of a jaspified function holds on to some memory, so one call per run would use up the memory over the thousands of runs in training. `q_batch` computes only the expectation values, for many photos at once, which is all we need to measure progress.

`gradient` puts the pieces together for one photo. `jax.vjp` evaluates the linear layer and also returns a function, `pullback`, which does the last step of the chain rule: given how the loss changes with each of the five angles, it returns how the loss changes with every entry of $M$ and $\underline{b}$. (The name vjp stands for vector-Jacobian product, which is what this function computes.) `jax.value_and_grad` differentiates the head, which gives the gradient for $\underline{v}$ and $c$ directly.

We train with Adam, taken from `optax`, a form of gradient descent that adapts the step size of every parameter to the size of its recent gradients. Each step uses the average gradient of four random training photos, a **minibatch**, instead of all 200. A step then costs $4 \times 31 = 124$ runs of the circuit instead of 6200.

Before Adam, we clip the gradient: if its length is above $1$, we scale it down to length $1$. Without this, training was unstable in our tests: runs with identical settings ended at test accuracies between 0.57 and 0.91. With clipping, and a smaller learning rate of 0.02 instead of 0.05, two runs both ended at 0.76.

Training takes a while, so the plot below is redrawn after every photo, with a line above it on how far training has got. The pale points, one per step, are the loss on each minibatch. It is noisy, because it comes from only four photos, each time different ones. Every five steps, we also compute the loss on 24 fixed training photos, shown as dots, which is the steadier measure of progress.

In [ ]:
def outputs(F, p, idx):
    """One prediction per image, from a single traced call."""
    X = jnp.stack([angles(F[i], p) for i in idx])
    q = np.asarray(q_batch(X, p["w"]))
    return np.array([float(head(jnp.array(q[k]), p)) for k in range(len(idx))])


def score(F, y, p, idx):
    yh = outputs(F, p, idx)
    return float(np.mean((yh - y[idx]) ** 2)), float(np.mean(np.sign(yh) == y[idx]))


batch_size, steps = 4, 40
sampler = np.random.default_rng(1)
tracked = sampler.choice(F_train.shape[0], 24, replace=False)

# Clip first: without it, runs with the same settings ended at very different accuracies.
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=0.02),
)
p = parameters
opt_state = optimizer.init(p)

loss, acc = score(F_train, y_train, p, tracked)
history = [(0, loss, acc)]
batch_losses = []
start_time = time.time()


def show_progress(photos_done):
    """Redraw the loss plot, with a line on how far training has got."""
    elapsed = time.time() - start_time
    total = steps * batch_size
    if photos_done < total:
        status = (f"step {photos_done // batch_size + 1} of {steps}, "
                  f"photo {photos_done % batch_size + 1} of {batch_size}, "
                  f"{elapsed / 60:.1f} min so far")
        if photos_done > 0:
            status += f", about {elapsed / photos_done * (total - photos_done) / 60:.0f} min to go"
    else:
        status = f"finished {steps} steps in {elapsed / 60:.1f} min"
    last_step, last_loss, last_acc = history[-1]
    status += f"\n24 tracked photos at step {last_step}: loss {last_loss:.3f}, accuracy {last_acc:.3f}"

    clear_output(wait=True)
    print(status)
    plt.figure(figsize=(6, 3.6))
    plt.plot(range(len(batch_losses)), batch_losses, marker=".", color="tab:blue", alpha=0.35,
             label=f"minibatch of {batch_size} photos")
    tracked_steps, tracked_losses, _ = zip(*history)
    plt.plot(tracked_steps, tracked_losses, marker="o", color="tab:blue", label="24 tracked photos")
    plt.xlim(0, steps)
    plt.ylim(bottom=0)
    plt.xlabel("training step")
    plt.ylabel("loss")
    plt.legend()
    plt.show()


for step in range(steps):
    batch = sampler.choice(F_train.shape[0], batch_size, replace=False)
    grads = None
    batch_loss = 0.0
    for k, i in enumerate(batch):
        show_progress(step * batch_size + k)
        value, g = gradient(F_train[i], y_train[i], p)
        batch_loss += value / batch_size
        grads = g if grads is None else jax.tree.map(jnp.add, grads, g)
    grads = jax.tree.map(lambda a: a / batch_size, grads)

    updates, opt_state = optimizer.update(grads, opt_state, p)
    p = optax.apply_updates(p, updates)
    batch_losses.append(batch_loss)

    if (step + 1) % 5 == 0:
        loss, acc = score(F_train, y_train, p, tracked)
        history.append((step + 1, loss, acc))

show_progress(steps * batch_size)

## Accuracy

The predicted class is the sign of $\hat{y}$: positive means cat, negative means dog. Measuring the accuracy needs no derivatives, so each photo costs one run of the circuit instead of 31. We check all 200 training photos and all 200 test photos.

In [ ]:
print("train accuracy:", score(F_train, y_train, p, np.arange(F_train.shape[0]))[1])
print("test  accuracy:", score(F_test,  y_test,  p, np.arange(F_test.shape[0]))[1])

## Saving and reusing the model

To classify a new photo, we need everything it passes through: the backbone, the standardisation, the linear layer, the circuit and the head. As in the QML tutorial, `save_model` stores the parts we name, and `load_model` returns them under the same names.

We do not store ResNet-18 itself. It is frozen, so it is exactly the network torchvision publishes, and we only record which weights we used, `IMAGENET1K_V1`. When loading, torchvision downloads the same weights again. This is why we named the weights explicitly at the start rather than taking torchvision's default, which may point to newer weights in a later release: with different weights, the features would change, and the trained linear layer would no longer fit them, without any error.

The standardisation is easy to forget. The 512 means and standard deviations come from our training photos, and a new photo has to be standardised with exactly the same numbers. We save them along with the encoding, the ansatz, the observables, the precision and all trained parameters.

In [ ]:
model_file = Path(tempfile.gettempdir()) / "pets_model.pkl"
save_model(
    model_file,
    backbone_weights=weights.name,
    mean=mean,
    std=std,
    encoding=encoding,
    ansatz=ansatz,
    observables=observables,
    precision=precision,
    parameters=p,
)

In a new session, we load the parts and rebuild the backbone from the recorded weights. The functions `angles` and `head` are the code of the two classical layers; their numbers come from the file. Like any format based on Python's pickle, a model file can run code when it is loaded, so only load files from a source you trust.

Below we classify four photos from the test part of the dataset, straight from the files on disk.

In [ ]:
saved = load_model(model_file)

saved_weights = ResNet18_Weights[saved["backbone_weights"]]
loaded_backbone = resnet18(weights=saved_weights)
loaded_backbone.fc = torch.nn.Identity()
loaded_backbone.eval()


def loaded_circuit(x, w):
    qf = QuantumFloat(saved["ansatz"].num_qubits)
    saved["encoding"].apply(qf, x)
    saved["ansatz"].apply(qf, w)
    return qf


loaded_block = expectation(loaded_circuit, saved["observables"], precision=saved["precision"])


@jaspify(terminal_sampling=True)
def loaded_q_batch(X, w):
    return jnp.stack([loaded_block(X[i], w) for i in range(X.shape[0])])


def classify(photos):
    """Predictions for a list of photos, from the saved model."""
    images = torch.stack([saved_weights.transforms()(photo.convert("RGB")) for photo in photos])
    with torch.no_grad():
        features = loaded_backbone(images).numpy()
    F = jnp.array((features - saved["mean"]) / saved["std"])
    trained = saved["parameters"]
    X = jnp.stack([angles(F[i], trained) for i in range(len(photos))])
    q = loaded_q_batch(X, trained["w"])
    return np.array([float(head(q[i], trained)) for i in range(len(photos))])


test_labels = np.array(test_data._bin_labels)
examples = np.concatenate([np.where(test_labels == c)[0][:2] for c in (0, 1)])
photos = [test_data[i][0] for i in examples]
predictions = classify(photos)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, photo, i, prediction in zip(axes, photos, examples, predictions):
    ax.imshow(photo.convert("RGB"))
    truth = "cat" if test_labels[i] == 0 else "dog"
    guess = "cat" if prediction > 0 else "dog"
    ax.set_title(f"{truth}: predicted {guess}\n" + r"$\hat{y}$" + f" = {prediction:+.2f}")
    ax.axis("off")
fig.tight_layout()
plt.show()

## What we have built

We classified photos of cats and dogs with a model made of a frozen classical network, a trained linear layer, a five-qubit circuit and a small head. ResNet-18 did the hardest part: it turned pixels into 512 features that already separate cats from dogs well, which is why a circuit of five qubits is enough on top.

The linear layer was trained through the circuit. The parameter-shift rule gave the derivatives of the circuit, and JAX's automatic differentiation carried them on into the linear layer. The cost per photo, 31 runs of the circuit, is set by the five angles and the ten ansatz parameters alone, however many features the linear layer takes in. The saved model holds everything a new photo needs except ResNet-18 itself, which is rebuilt from its published weights.

Two limits are worth keeping in mind. The expectation values are estimated from measurements, so every gradient carries noise, and the numbers above change a little from run to run. And on these features, a classical layer of the same size in place of the circuit would do about as well. This tutorial shows how to train a quantum model inside a larger classical one; it makes no claim that the quantum part classifies better.

The [hybrid model example](../../reference/Examples/HybridQMLModel.rst) shows how to pack the parameters of several classical and quantum parts into one vector, and the [QML tutorial](./QMLtutorial.ipynb) introduces the primitives themselves.